# Lesson 5.1: Confidence Intervals

Build a 95% z-interval for a mean when sigma is known, then simulate thousands of intervals to see about 95% catch the true mean.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. The z-interval by formula

The lesson's exam: sigma = 12 is known, n = 40 and the sample mean is 78.5. Three steps: standard error, margin of error (z* times SE), then the interval.

In [1]:
import numpy as np
from scipy import stats

sigma, n, xbar = 12, 40, 78.5
z = stats.norm.ppf(0.975)
se = sigma / np.sqrt(n)
me = z * se

print(f'z* = {z:.3f}, SE = {se:.3f}, margin = {me:.2f}')
print(f'95% CI = ({xbar - me:.2f}, {xbar + me:.2f})')

## 2. Confidence level changes the width

Being more confident means casting a wider net. We repeat the calculation for 90%, 95% and 99%, using the matching z*.

In [2]:
for level in (0.90, 0.95, 0.99):
    z = stats.norm.ppf(1 - (1 - level) / 2)
    me = z * se
    print(f'{level:.0%}: z* = {z:.3f}, margin = {me:.2f}, ({xbar - me:.2f}, {xbar + me:.2f})')

## 3. Sample size changes the width

At 95%, quadrupling n halves the margin of error, the 1 over square root of n rule again.

In [3]:
for n_try in (10, 40, 160, 640):
    margin = 1.96 * sigma / np.sqrt(n_try)
    print(f'n = {n_try:3d}: margin = {margin:.2f}')

## 4. Does it really catch the truth?

Make the truth known: a normal population with mean 78.5 and SD 12. Draw 10,000 samples of 40, build a 95% interval from each, and count how many contain 78.5.

In [4]:
rng = np.random.default_rng(1)
true_mean = 78.5
means = rng.normal(true_mean, sigma, size=(10_000, 40)).mean(axis=1)
me = 1.96 * sigma / np.sqrt(40)
caught = (means - me <= true_mean) & (true_mean <= means + me)

print(f'intervals that caught the true mean: {caught.mean():.1%}')
print('intervals that missed:', (~caught).sum(), 'of 10000')

## Try this

- Change the confidence level to 99% in the first block and predict whether the interval gets wider or narrower.
- Change sigma to 6 and predict what happens to the margin of error.
- In the simulation, use a z* of 1.645 (a 90% interval) in place of 1.96. What share of intervals should catch the truth now?

## The same idea in R, SPSS and Stata

**R**

```r
sigma <- 12; n <- 40; xbar <- 78.5
z <- qnorm(0.975)
me <- z * sigma / sqrt(n)
c(z, me, xbar - me, xbar + me)
set.seed(1)
means <- replicate(10000, mean(rnorm(40, 78.5, 12)))
mean(abs(means - 78.5) <= me)
```

**SPSS** (Transform > Compute Variable (use IDF.NORMAL in the function list))

```
DATA LIST FREE / sigma n xbar.
BEGIN DATA
12 40 78.5
END DATA.
COMPUTE z = IDF.NORMAL(0.975, 0, 1).
COMPUTE se = sigma / SQRT(n).
COMPUTE me = z * se.
COMPUTE lower = xbar - me.
COMPUTE upper = xbar + me.
LIST VARIABLES=z se me lower upper.
```

**Stata**

```
scalar z = invnormal(0.975)
scalar se = 12 / sqrt(40)
scalar me = z * se
display "z* = " z
display "SE = " se
display "margin = " me
display "95% CI = (" (78.5 - me) ", " (78.5 + me) ")"
```